In [12]:
import sys
from pathlib import Path

# I had problems with relative imports in Jupyter notebooks, so I added the project root to sys.path
PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Added to sys.path:", PROJECT_ROOT)

Added to sys.path: /Users/francescoorma99/Desktop/Francesco/Denmark/DTU/Classes/2 semester/Renewable in Electricity Market/assignment1


In [13]:
from pathlib import Path
import pandas as pd

from src.data_loader import load_rts24_from_csv
from src.models import Step1InputData, Step1MarketClearing
from src.postprocess import step1_build_summary_tables

Define data directory

In [14]:
DATA_DIR = Path("..") / "data"

data = load_rts24_from_csv(DATA_DIR)

print(f"Slack bus: {data.slack_bus}")
print(f"Number of generators: {len(data.generators)}")
print(f"Number of lines: {len(data.lines)}")

Slack bus: 13
Number of generators: 12
Number of lines: 34


# STEP 1

Choose hour for copper-plate market clearing

In [15]:
H = 19  # peak hour example

Extract hour-specific data

In [16]:
# Generators
gen_df = data.generators.copy()

GENERATORS = gen_df["unit"].tolist()
generator_cost = dict(zip(gen_df["unit"], gen_df["Ci_USD_per_MWh"]))
generator_pmax = dict(zip(gen_df["unit"], gen_df["Pmax_MW"]))


# Wind availability
wind_farms_df = data.wind_farms.copy()
wind_av_h = data.wind_availability.query("hour == @H")

WINDS = wind_farms_df["wind_id"].tolist()
wind_avail = dict(zip(wind_av_h["wind_id"], wind_av_h["available_MW"]))


# Nodal demand
nodal_load_h = data.nodal_load.query("hour == @H")

LOAD_BUSES = nodal_load_h.query("demand_MW > 0")["bus"].tolist()
demand_pmax = dict(zip(nodal_load_h["bus"], nodal_load_h["demand_MW"]))


# Demand bids
bids_h = data.demand_bids.query("hour == @H")
demand_bid = dict(zip(bids_h["bus"], bids_h["bid_price_USD_per_MWh"]))

Build input object

In [17]:
inp = Step1InputData(
    GENERATORS=GENERATORS,
    WINDS=WINDS,
    LOAD_BUSES=LOAD_BUSES,
    generator_cost=generator_cost,
    generator_pmax=generator_pmax,
    wind_avail=wind_avail,
    demand_pmax=demand_pmax,
    demand_bid=demand_bid,
)

Solve Step 1 (Copper Plate Market Clearing)

In [18]:
model = Step1MarketClearing(inp, output_flag=0)
model.run()

Results

In [19]:
df_gen, df_wind, df_dem, totals = step1_build_summary_tables(model.results, inp)

print("Market price:", round(totals["market_price"], 2), "USD/MWh")
print("Objective (Social Welfare):", round(totals["objective_welfare"], 2))

print("\nTotal generation:", round(totals["total_gen_MW"], 2), "MW")
print("Total wind:", round(totals["total_wind_MW"], 2), "MW")
print("Total demand served:", round(totals["total_demand_served_MW"], 2), "MW")

Market price: 13.32 USD/MWh
Objective (Social Welfare): 619868.95

Total generation: 2169.19 MW
Total wind: 481.31 MW
Total demand served: 2650.5 MW


Generator results

In [20]:
df_gen

,generator,p_MW,marginal_cost,profit
0,1,99.185514,13.32,0.0
1,2,0.000000,13.32,0.0
2,3,0.000000,20.70,-0.0
3,4,0.000000,20.93,-0.0
4,5,0.000000,26.11,-0.0
5,6,155.000000,10.52,434.0
6,7,155.000000,10.52,434.0
7,8,400.000000,6.02,2920.0
8,9,400.000000,5.47,3140.0
9,10,300.000000,0.00,3996.0


Wind results

In [ ]:
df_wind

Demand results

In [21]:
df_dem

,bus,d_MW,bid_price,utility
0,1,100.7190,240.0,22830.98292
1,2,90.1170,240.0,20427.72156
2,3,166.9815,240.0,37851.36642
3,4,68.9130,240.0,15621.19884
4,5,66.2625,240.0,15020.38350
5,6,127.2240,240.0,28839.13632
6,7,116.6220,240.0,26435.87496
7,8,159.0300,240.0,36048.92040
8,9,161.6805,240.0,36649.73574
9,10,180.2340,240.0,40855.44312


Welfare breakdown check

In [22]:
print("Total generator profit:", round(totals["total_gen_profit"], 2))
print("Total wind profit:", round(totals["total_wind_profit"], 2))
print("Total demand utility:", round(totals["total_demand_utility"], 2))

print("\nCheck: Utility - Cost = Welfare")
print(round(
    totals["total_demand_utility"]
    + totals["total_gen_profit"]
    + totals["total_wind_profit"],
    6
))

Total generator profit: 12642.5
Total wind profit: 6411.11
Total demand utility: 600815.34

Check: Utility - Cost = Welfare
619868.948948
